# AmbiFix: new original-only BERT — Mac M3 Pro

Separate MPS FP32 experiment, not recovery of the historical model or an identical CUDA/BF16 run. Original training4731; validation1015. No final evaluation. Read README_MAC.md. Run All once after freeing adequate storage. This notebook delegates heavy execution to a fresh isolated Python process. Genuine preflight failures stop training. No hyperparameter search or repeated score-chasing trials.


In [ ]:
from pathlib import Path
import json, os, sys, subprocess
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "data_manifest.json").is_file() and (p / "scripts/mac_preflight.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the extracted Mac package, keeping its files together.")
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from verify_inputs import verify_inputs
checked = verify_inputs(ROOT)
print("Frozen original training:", len(checked["train_rows"]), "validation:", len(checked["validation_rows"]))
print("Class order:", checked["config"]["dataset_class_names"])
print("Train-only square-root weights:", checked["class_weights"])
print("Only CPU input checks have run. No BERT model or held-out data has been loaded.")


## Isolated environment

The setup requires16GiB actually free before installation;20GiB or more is recommended. At least12GiB must remain afterward. A failed storage check installs nothing and starts no training. Existing experiment environments are never changed.


In [ ]:
ENV_PYTHON = ROOT / ".venv/bin/python"
if not ENV_PYTHON.is_file():
    subprocess.run([sys.executable, str(ROOT / "scripts/setup_mac_environment.py")], cwd=ROOT, check=True)
if not ENV_PYTHON.is_file():
    raise RuntimeError("The dedicated Mac environment is missing. Read the setup report; training is blocked.")
print("Dedicated environment:", ENV_PYTHON)


## Single fixed run

This command performs the train-only BERT forward/backward/save/reload smoke test, resets/reinitializes from the pinned original pretrained checkpoint, then trains only if all checks pass. No CPU fallback; FP32 MPS. Validation Macro F1 selects the earliest literal best. An interrupted run is retained, not silently retrained. The complete model and actual validation figures are exported only after reload and independent verification succeed.


In [ ]:
run_env = dict(os.environ)
run_env.update({"PYTHONHASHSEED":"42", "PYTORCH_ENABLE_MPS_FALLBACK":"0", "PYTORCH_MPS_FAST_MATH":"0", "PYTORCH_MPS_PREFER_METAL":"0"})
run_env.pop("PYTHONOPTIMIZE", None)
subprocess.run([str(ENV_PYTHON), str(ROOT / "scripts/train_bert.py"), "--execute"], cwd=ROOT, env=run_env, check=True)
completion = json.loads((ROOT / "outputs/COMPLETED.json").read_text())
if completion["status"] != "COMPLETED_NEW_VALIDATION_ONLY_RUN":
    raise RuntimeError("A complete verified result was not produced.")
print("ACTUAL validation Macro F1:", completion["selected_validation_macro_f1"])
print("ACTUAL selected epoch:", completion["selected_epoch"])


## Return and freeze

Save this executed notebook. Preserve the full actual COMPLETE_ARTIFACTS.zip and matching outputs/COMPLETED.json. The package-root preparation status is a historical audit snapshot, not training success. Independently inspect the returned checkpoint/results before freezing. A separate explicit authorization is required before any final held-out evaluation; that guarded script is never called here.


In [ ]:
from IPython.display import display, FileLink
for relative in [completion["return_zip"], "outputs/COMPLETED.json"]:
    path = ROOT / relative
    if not path.is_file():
        raise FileNotFoundError("Required actual result missing: " + str(path))
    print(path)
    display(FileLink(str(path)))
print("Save this notebook with its real outputs. No held-out evaluation has run.")
